In [ ]:
# =============================================================================
# CELL 1: Environment Setup & Package Dependencies
# =============================================================================
!pip install -q transformers accelerate bitsandbytes nltk pandas pillow requests pycocotools tqdm

import gc, io, json, os, random, time
import matplotlib.pyplot as plt
import nltk
from nltk.tokenize import word_tokenize
import numpy as np
import pandas as pd
from PIL import Image
import requests
import torch
from transformers import AutoProcessor, AutoModelForCausalLM, LlavaForConditionalGeneration
from tqdm.auto import tqdm
from google.colab import files

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

print("✅ CELL 1 complete: dependencies ready, seed fixed at", RANDOM_SEED)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 16.4 MB/s eta 0:00:00
✅ CELL 1 complete: dependencies ready, seed fixed at 42


In [ ]:
# =============================================================================
# CELL 2: MSCOCO 2014 Download & Fixed 500-Image Sampler
# =============================================================================
# WHY: Using the SAME random seed (42) + SAME sampling code guarantees this
# produces the IDENTICAL 500 image IDs in every notebook — that's what
# keeps all 3 models comparable even when run in totally separate sessions.
# TIME: ~2-5 minutes.
# =============================================================================
ANNOTATIONS_URL = "http://images.cocodataset.org/annotations/annotations_trainval2014.zip"
ANNOTATIONS_ZIP = "annotations_trainval2014.zip"
CAPTIONS_JSON = "annotations/captions_val2014.json"

if not os.path.exists(CAPTIONS_JSON):
    print("⏬ Downloading REAL MSCOCO 2014 validation annotations...")
    os.system(f"wget -q {ANNOTATIONS_URL}")
    os.system(f"unzip -q {ANNOTATIONS_ZIP}")

with open(CAPTIONS_JSON, "r") as f:
    coco_data = json.load(f)

image_id_to_refs = {}
for ann in coco_data["annotations"]:
    img_id = str(ann["image_id"]).zfill(12)
    image_id_to_refs.setdefault(img_id, []).append(ann["caption"])

all_image_ids = sorted(list(image_id_to_refs.keys()))
random.seed(RANDOM_SEED)
SAMPLE_SIZE = 500
sampled_image_ids = random.sample(all_image_ids, SAMPLE_SIZE)

def get_coco_2014_image_url(image_id):
    return f"http://images.cocodataset.org/val2014/COCO_val2014_{image_id}.jpg"

print(f"✅ Sampled {len(sampled_image_ids)} FIXED real COCO images (seed={RANDOM_SEED})")
print(f"   First 3 IDs (should match across all 3 notebooks): {sampled_image_ids[:3]}")

⏬ Downloading REAL MSCOCO 2014 validation annotations...
✅ Sampled 500 FIXED real COCO images (seed=42)
   First 3 IDs (should match across all 3 notebooks): ['000000105156', '000000022861', '000000258529']


In [ ]:
# =============================================================================
# CELL 3: Download & Cache the 500 Real Images (with progress bar)
# =============================================================================
SMOKE_TEST = False  # ⚠️ Set to False before your real run
RUN_IMAGE_IDS = sampled_image_ids[:10] if SMOKE_TEST else sampled_image_ids
print(f"🚀 Preparing {len(RUN_IMAGE_IDS)} images (Smoke Test = {SMOKE_TEST})")

image_cache = {}
t0 = time.time()
for img_id in tqdm(RUN_IMAGE_IDS, desc="⏬ Downloading real COCO images", unit="img"):
    url = get_coco_2014_image_url(img_id)
    resp = requests.get(url, timeout=15)
    image_cache[img_id] = Image.open(io.BytesIO(resp.content)).convert("RGB")

print(f"✅ Cached {len(image_cache)} real images in {time.time()-t0:.1f}s")

🚀 Preparing 500 images (Smoke Test = False)


⏬ Downloading real COCO images:   0%|          | 0/500 [00:00<?, ?img/s]

✅ Cached 500 real images in 626.7s


In [ ]:
# =============================================================================
# CELL 4: 4-Bit Quantized VLM Pipeline + Reusable Runner (with progress bar)
# =============================================================================
from transformers import BitsAndBytesConfig

BASELINE_PROMPT = "Provide a concise description of this image."
GROUNDED_PROMPT = "Describe this image concisely. Mention only visually supported objects, attributes, actions, and relationships. Omit any uncertain or unverified details."

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

MODEL_CONFIGS = [
    {"name": "LLaVA-1.5-7B",  "path": "llava-hf/llava-1.5-7b-hf",     "type": "llava"},
    {"name": "Qwen2.5-VL-3B", "path": "Qwen/Qwen2.5-VL-3B-Instruct",  "type": "qwen2_5_vl"},
    {"name": "Qwen3-VL-4B",   "path": "Qwen/Qwen2-VL-7B-Instruct",    "type": "qwen2_vl"},
]


def load_model_and_processor(config):
    print(f"📦 Loading {config['name']} in 4-bit precision...")
    t0 = time.time()
    processor = AutoProcessor.from_pretrained(config["path"])

    if config["type"] == "llava":
        model_class = LlavaForConditionalGeneration
    elif config["type"] == "qwen2_5_vl":
        from transformers import Qwen2_5_VLForConditionalGeneration
        model_class = Qwen2_5_VLForConditionalGeneration
    elif config["type"] == "qwen2_vl":
        from transformers import Qwen2VLForConditionalGeneration
        model_class = Qwen2VLForConditionalGeneration
    else:
        raise ValueError(f"Unknown model type: {config['type']}")

    model = model_class.from_pretrained(
        config["path"], quantization_config=bnb_config,
        device_map="auto", low_cpu_mem_usage=True,
    )
    print(f"✅ Loaded in {time.time()-t0:.1f}s")
    return model, processor


def generate_vlm_caption(model, processor, image, prompt_text, model_type="llava"):
    if model_type == "llava":
        formatted_prompt = f"USER: <image>\n{prompt_text}\nASSISTANT:"
        inputs = processor(text=formatted_prompt, images=image, return_tensors="pt").to("cuda")
        with torch.no_grad():
            generate_ids = model.generate(**inputs, max_new_tokens=100, do_sample=False)
        prompt_len = inputs["input_ids"].shape[1]
        caption = processor.batch_decode(
            generate_ids[:, prompt_len:], skip_special_tokens=True, clean_up_tokenization_spaces=True
        )[0].strip()
    else:
        messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt_text}]}]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text], images=[image], return_tensors="pt").to("cuda")
        with torch.no_grad():
            generate_ids = model.generate(**inputs, max_new_tokens=100, do_sample=False)
        trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generate_ids)]
        caption = processor.batch_decode(
            trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=True
        )[0].strip()
    return caption


def flush_gpu_memory(model, processor):
    del model, processor
    gc.collect()
    torch.cuda.empty_cache()
    print("🧹 Flushed GPU VRAM.")


def run_single_model_experiment(config, run_image_ids, image_cache, image_id_to_refs, checkpoint_every=25):
    """
    Runs ONE model on all 500 real images, with a LIVE progress bar showing
    elapsed time + ETA, and checkpoints to CSV every 25 images so partial
    progress is never lost even if Colab disconnects mid-run.
    """
    model, processor = load_model_and_processor(config)
    records = []
    csv_path = f"captions_{config['name']}.csv"
    t_start = time.time()

    progress_bar = tqdm(run_image_ids, desc=f"🔄 {config['name']}", unit="img")
    for i, img_id in enumerate(progress_bar):
        img = image_cache[img_id]
        refs = image_id_to_refs[img_id]

        base_cap = generate_vlm_caption(model, processor, img, BASELINE_PROMPT, config["type"])
        ground_cap = generate_vlm_caption(model, processor, img, GROUNDED_PROMPT, config["type"])

        records.append({
            "image_id": img_id,
            "model": config["name"],
            "baseline_caption": base_cap,
            "grounded_caption": ground_cap,
            "reference_captions": " | ".join(refs),
        })

        elapsed = time.time() - t_start
        avg_per_img = elapsed / (i + 1)
        eta_min = (avg_per_img * (len(run_image_ids) - i - 1)) / 60
        progress_bar.set_postfix(elapsed_min=f"{elapsed/60:.1f}", eta_min=f"{eta_min:.1f}")

        if (i + 1) % checkpoint_every == 0 or (i + 1) == len(run_image_ids):
            pd.DataFrame(records).to_csv(csv_path, index=False)

    total_min = (time.time() - t_start) / 60
    print(f"✅ {config['name']} finished {len(records)} real images in {total_min:.1f} minutes")
    print(f"💾 CSV saved locally: {csv_path}")

    flush_gpu_memory(model, processor)
    return pd.DataFrame(records), csv_path


print("✅ CELL 4 complete: model engine ready.")

✅ CELL 4 complete: model engine ready.


In [ ]:
# =============================================================================
# CELL 5: RUN THIS MODEL, THEN AUTO-DOWNLOAD ITS CSV
# =============================================================================
# In Notebook 1 -> MODEL_CONFIGS[0]  (LLaVA-1.5-7B)
# In Notebook 2 -> MODEL_CONFIGS[1]  (Qwen2.5-VL-3B)
# In Notebook 3 -> MODEL_CONFIGS[2]  (Qwen3-VL-4B)
# =============================================================================
config = MODEL_CONFIGS[0]   # 👈 CHANGE THIS INDEX PER NOTEBOOK (0, 1, or 2)

try:0
    df_result, csv_path = run_single_model_experiment(
        config, RUN_IMAGE_IDS, image_cache, image_id_to_refs
    )
    print(f"\n⬇️ Triggering browser download for {csv_path} ...")
    files.download(csv_path)   # <-- this pops the CSV into your Downloads folder
    print("✅ Done. Save this CSV somewhere safe — you'll upload it in the combiner notebook.")
except Exception as e:
    print(f"⚠️ Could not run {config['name']}: {e}")

📦 Loading LLaVA-1.5-7B in 4-bit precision...


processor_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/674 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/505 [00:00<?, ?B/s]

...

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


✅ LLaVA-1.5-7B finished 500 real images in 140.4 minutes
💾 CSV saved locally: captions_LLaVA-1.5-7B.csv
🧹 Flushed GPU VRAM.

⬇️ Triggering browser download for captions_LLaVA-1.5-7B.csv ...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Done. Save this CSV somewhere safe — you'll upload it in the combiner notebook.
